Notebook de pruebas:
Tiene todas las funcionalidades del standalone (main_despliegue_standalone)

Nota: Si este notebook da errores de importación asociados a Cartopy (CRSS); se debe reiniciar el kernel y volver a ejecutar todo

In [ ]:
####################### N O  T O C A R ############################################
%reload_ext autoreload
%autoreload 2

import os
import sys
import pandas as pd

root_path = os.path.abspath(os.path.join(os.path.dirname(os.getcwd()), ".."))
sys.path.append(root_path)


# Importar configuraciones del modulo
from procesado_datos.config_modulo.config_procesado import config_modulo_procesado
# Importar configuraciones del submodulo de despliegue
from procesado_datos.transmision.configs.configuracion_transmision import config_transmision

# Importar manager de configuraciones
from procesado_datos.config_modulo.ProcesadoConfig import ProcesadoConfig

# Importar servicios necesarios
from pathlib import Path
import procesado_datos.services.Carga.cargar_datos_csv as carga
from procesado_datos.services.Utils.utilidades import *
from procesado_datos.services.Correctores.corrector_utils import *
from procesado_datos.services.Graficado.graficar_series_y_guardar import graficar_series_y_guardar
from procesado_datos.services.Graficado.graficar_mapa_de_posiciones import graficar_mapa_de_posiciones
from procesado_datos.services.Utils.excel_a_png import *
from procesado_datos.transmision.configs import configuracion_transmision

##################################################################################

In [ ]:
# Crear la instancia del manager de configuraciones
config = ProcesadoConfig.from_sources(
    config_modulo_procesado,
    config_transmision,
)    

In [ ]:
config.variables_a_graficar

In [ ]:
rutas_de_sondas, seriales_encontrados = carga.buscar_nombre_de_archivo_de_sonda(config)
rutas_de_sondas[0]

In [ ]:
# 7. Guarda las pruebas de laboratorio en un archivo png
ruta_a_carpeta = os.path.join(config["carpeta_de_guardado_de_datos_procesados"])
fecha_del_estudio = config.convertir_a_pd_datetime("fecha_del_estudio", formato="%Y-%m-%d")
carpeta_del_estudio = f"{fecha_del_estudio.year:04d}{fecha_del_estudio.month:02d}"
ruta_a_la_carpeta_de_guardado = os.path.join(ruta_a_carpeta, carpeta_del_estudio)

csv_a_png(rutas_de_sondas[0],
     carpeta_salida= ruta_a_la_carpeta_de_guardado,
     max_filas= 20,
     nombre_salida= "mensaje.png"
)

print(f"Se ha guardado el png del mensaje de transmision")

In [ ]:
diccionario_de_datos_de_sondas = carga.cargar_datos_de_sonda(rutas_de_sondas, seriales_encontrados)

In [ ]:
diccionario_de_datos_de_sondas.keys()

In [ ]:
diccionario_de_sondas_en_fechas = carga.seleccionar_rango_de_fechas(diccionario = diccionario_de_datos_de_sondas, buscar_fechas_anteriores_al_estudio = False, config = config)

In [ ]:
datos_de_sondas_sin_duplicados = carga.buscar_y_eliminar_duplicados(diccionario_de_sondas_en_fechas)

In [ ]:
datos_ordenados = carga.ordernar_datos_por_fecha(diccionario_de_sondas_en_fechas)

In [ ]:
datos_ordenados.keys()

In [ ]:
for serial in seriales_encontrados:
    if serial in datos_ordenados:
        datos_ordenados[serial]["tspan_rounded"] = datos_ordenados[serial]["tspan_de_envio"]


In [ ]:
# Eliminar datos espurios (solo se revisa si hay valores de rapidez superiores a 2 m/s y se elimina toda la fila)
datos_finales = eliminar_datos_espurios(datos_ordenados)

In [ ]:
# Agregar componentes de la velocidad al diccionario con los dataframe de cada sonda
datos_finales = carga.agregar_componentes_de_la_velocidad(datos_finales)

In [ ]:
datos_finales["4912208"].columns

In [ ]:
# Crear ruta a la carpeta de guardado de datos de laboratorio
ruta_a_carpeta = config.carpeta_de_guardado_de_datos_procesados 
fecha_del_estudio = config.convertir_a_pd_datetime("fecha_del_estudio", formato="%Y-%m-%d")
carpeta_del_estudio = f"{fecha_del_estudio.year:04d}{fecha_del_estudio.month:02d}"
ruta_a_la_carpeta_de_guardado = os.path.join(ruta_a_carpeta, carpeta_del_estudio)

In [ ]:
ruta_a_la_carpeta_de_guardado

In [ ]:
# Guardar datos del estudio en archivo .pkl
carpeta_de_destino = ruta_a_la_carpeta_de_guardado
nombre_de_archivo = config.nombre_del_archivo_de_datos_procesados
guardar_diccionario_como_pickle(diccionario = datos_finales, 
                                ruta = carpeta_de_destino, 
                                nombre_archivo=nombre_de_archivo)


In [ ]:
tabla_de_porcentajes = calcular_porcentaje_de_datos_recibidos(datos_finales, config)

In [ ]:
tabla_de_porcentajes

In [ ]:
# Guardar Tabla de porcentajes
guardar_porcentajes_en_excel(data= tabla_de_porcentajes, ruta= ruta_a_la_carpeta_de_guardado, nombre_de_archivo=config.nombre_del_excel_de_porcentajes)

In [ ]:
graficar_series_y_guardar(
    datos= datos_finales,
    ruta_a_carpeta_de_guardado=ruta_a_la_carpeta_de_guardado, 
    mostrar_figura=False, 
    config=config
)

In [ ]:
graficar_mapa_de_posiciones(
    datos=datos_finales, 
    ruta_a_la_carpeta_de_guardado = ruta_a_la_carpeta_de_guardado,
    mostrar_figura=False,
    config=config)